# Data Exploration - Yelp Dataset

This notebook explores the Yelp dataset for the Tourist Recommendation System.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data import YelpDataLoader, TextPreprocessor, DataSampler
from src.config import get_config

# Setup
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
config = get_config()

## 1. Load the Data

In [ ]:
# Initialize loader
loader = YelpDataLoader()

# Load a sample of the data (adjust limit as needed)
# For tourism, filter to relevant categories
tourism_categories = [
    "Restaurant", "Hotel", "Museum", "Park", "Bar",
    "Cafe", "Attraction", "Entertainment", "Shopping"
]

# Load with limit for exploration
data = loader.load_all(limit=10000, categories_filter=tourism_categories)

# Get stats
stats = loader.get_dataset_stats(data)
for name, info in stats.items():
    print(f"\n{name.upper()}:")
    print(f"  Count: {info['count']:,}")
    print(f"  Memory: {info['memory_mb']:.2f} MB")

## 2. Explore Business Data

In [ ]:
businesses = data['businesses']
print(f"Columns: {businesses.columns.tolist()}")
businesses.head()

In [ ]:
# Star rating distribution
if 'stars' in businesses.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    businesses['stars'].hist(bins=9, ax=ax, edgecolor='black')
    ax.set_xlabel('Star Rating')
    ax.set_ylabel('Count')
    ax.set_title('Business Star Rating Distribution')
    plt.tight_layout()
    plt.show()

In [ ]:
# Top categories
if 'categories' in businesses.columns:
    all_cats = []
    for cats in businesses['categories'].dropna():
        all_cats.extend([c.strip() for c in cats.split(',')])
    
    cat_counts = Counter(all_cats).most_common(20)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    cats, counts = zip(*cat_counts)
    ax.barh(cats[::-1], counts[::-1])
    ax.set_xlabel('Count')
    ax.set_title('Top 20 Business Categories')
    plt.tight_layout()
    plt.show()

## 3. Explore Review Data

In [ ]:
reviews = data['reviews']
print(f"Columns: {reviews.columns.tolist()}")
reviews.head()

In [ ]:
# Review length distribution
if 'text' in reviews.columns:
    reviews['text_length'] = reviews['text'].str.len()
    reviews['word_count'] = reviews['text'].str.split().str.len()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    reviews['text_length'].hist(bins=50, ax=axes[0], edgecolor='black')
    axes[0].set_xlabel('Character Count')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Review Length Distribution (Characters)')
    
    reviews['word_count'].hist(bins=50, ax=axes[1], edgecolor='black')
    axes[1].set_xlabel('Word Count')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Review Length Distribution (Words)')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Average review length: {reviews['word_count'].mean():.1f} words")
    print(f"Median review length: {reviews['word_count'].median():.1f} words")

In [ ]:
# Reviews per user distribution
reviews_per_user = reviews['user_id'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
reviews_per_user.hist(bins=50, ax=ax, edgecolor='black')
ax.set_xlabel('Number of Reviews')
ax.set_ylabel('Number of Users')
ax.set_title('Reviews per User Distribution')
ax.set_xlim(0, reviews_per_user.quantile(0.95))  # Limit to 95th percentile
plt.tight_layout()
plt.show()

print(f"Average reviews per user: {reviews_per_user.mean():.2f}")
print(f"Users with 5+ reviews: {(reviews_per_user >= 5).sum():,}")

## 4. Text Preprocessing Demo

In [ ]:
preprocessor = TextPreprocessor()

# Sample some reviews and clean them
sample_reviews = reviews['text'].head(5).tolist()

for i, review in enumerate(sample_reviews):
    cleaned = preprocessor.clean_text(review)
    print(f"--- Review {i+1} ---")
    print(f"Original ({len(review)} chars): {review[:200]}...")
    print(f"Cleaned ({len(cleaned)} chars): {cleaned[:200]}...")
    print()

In [ ]:
# Process all reviews
reviews_clean = preprocessor.process_dataframe(reviews, text_column='text')
print(f"Cleaned {len(reviews_clean)} reviews")
reviews_clean[['text', 'clean_text']].head()

## 5. Data Sampling Demo

In [ ]:
sampler = DataSampler()

# Filter by activity
reviews_filtered = sampler.filter_by_activity(
    reviews_clean,
    min_user_reviews=5,
    min_business_reviews=3
)

print(f"After filtering: {len(reviews_filtered)} reviews")
print(f"Unique users: {reviews_filtered['user_id'].nunique()}")
print(f"Unique businesses: {reviews_filtered['business_id'].nunique()}")

In [ ]:
# Train/val/test split
train, val, test = sampler.train_val_test_split(reviews_filtered)

print(f"Train: {len(train)} ({len(train)/len(reviews_filtered)*100:.1f}%)")
print(f"Val: {len(val)} ({len(val)/len(reviews_filtered)*100:.1f}%)")
print(f"Test: {len(test)} ({len(test)/len(reviews_filtered)*100:.1f}%)")

## 6. Save Processed Data

In [ ]:
from src.config import PROCESSED_DATA_DIR

# Save processed datasets
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train.to_parquet(PROCESSED_DATA_DIR / 'train_reviews.parquet', index=False)
val.to_parquet(PROCESSED_DATA_DIR / 'val_reviews.parquet', index=False)
test.to_parquet(PROCESSED_DATA_DIR / 'test_reviews.parquet', index=False)
businesses.to_parquet(PROCESSED_DATA_DIR / 'businesses.parquet', index=False)

print("Saved processed data to:", PROCESSED_DATA_DIR)

## Next Steps

1. **Phase 2**: Train BERT MBTI classifier on user reviews
2. **Phase 3**: Apply BERTopic for venue topic modeling
3. **Phase 4**: Build heterogeneous graph and train GNN